# BONUS ABLATION: No Positional Embeddings (RoPE removed)
Retraining Model H (Hindi) from scratch with Rotary Position Embeddings completely removed.
All other architectural choices (8 layers, 512 dim, 8 heads, 1024 d_ff, LayerNorm, SwiGLU, Weight Tying) remain identical.

In [1]:
import os
import math
import time
import json
import random
import logging
from dataclasses import dataclass
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ==========================================
# UPDATE THESE PATHS FOR KAGGLE
# ==========================================
TRAIN_BIN = "/kaggle/input/notebooks/srinivasu2005/tokenize-hindi/hindi_tokenized/train.bin"
VAL_BIN = "/kaggle/input/notebooks/srinivasu2005/tokenize-hindi/hindi_tokenized/val.bin"
OUTPUT_DIR = "/kaggle/working/checkpoints_hindi_no_rope"

# ==========================================
# CONFIGURATION
# ==========================================
VOCAB_SIZE = 8000
D_MODEL = 512
N_LAYERS = 8
N_HEADS = 8
D_FF = 1024
MAX_SEQ_LEN = 512
DROPOUT = 0.1
TIE_WEIGHTS = True
PAD_ID = 0; BOS_ID = 2; EOS_ID = 3

# Training config
LR = 3e-4
MIN_LR = 1e-5
WARMUP_STEPS = 2000
MAX_STEPS = 16000
BATCH_SIZE = 16
GRAD_ACCUM = 4
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
SEED = 42
EVAL_INTERVAL = 500
SAVE_INTERVAL = 1000
LOG_INTERVAL = 50
EVAL_MAX_BATCHES = 100

os.makedirs(OUTPUT_DIR, exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] %(levelname)s — %(message)s",
    handlers=[
        logging.FileHandler(os.path.join(OUTPUT_DIR, "train_no_rope.log")),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("train_hindi_no_rope")

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(SEED)

In [2]:
@dataclass
class TransformerConfig:
    vocab_size: int = VOCAB_SIZE
    d_model: int = D_MODEL
    n_layers: int = N_LAYERS
    n_heads: int = N_HEADS
    d_ff: int = D_FF
    max_seq_len: int = MAX_SEQ_LEN
    dropout: float = DROPOUT
    tie_weights: bool = TIE_WEIGHTS
    pad_id: int = PAD_ID; bos_id: int = BOS_ID; eos_id: int = EOS_ID
    
    @property
    def d_k(self):
        return self.d_model // self.n_heads

config = TransformerConfig()

In [3]:
# ==========================================
# MODEL ARCHITECTURE (NO ROPE)
# ==========================================

class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads, self.d_k, self.d_model = cfg.n_heads, cfg.d_k, cfg.d_model
        self.W_Q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_K = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_V = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_O = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)
        # NO RoPE buffers - only causal mask
        mask = torch.triu(torch.ones(cfg.max_seq_len, cfg.max_seq_len), diagonal=1).bool()
        self.register_buffer("causal_mask", mask)

    def forward(self, x, return_attention=False):
        B, T, d = x.shape
        Q = self.W_Q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        
        # Standard scaled dot-product WITHOUT positional rotation
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        scores = scores.masked_fill(self.causal_mask[:T, :T].unsqueeze(0).unsqueeze(0), float("-inf"))
        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights_d = self.attn_dropout(attn_weights)
        
        context = torch.matmul(attn_weights_d, V)
        context = context.transpose(1, 2).contiguous().view(B, T, self.d_model)
        output = self.resid_dropout(self.W_O(context))
        return (output, attn_weights) if return_attention else (output, None)

class SwiGLUFeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.W_gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.W_up = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.W_down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)
        
    def forward(self, x):
        return self.dropout(self.W_down(F.silu(self.W_gate(x)) * self.W_up(x)))

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = MultiHeadCausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = SwiGLUFeedForward(cfg)
        
    def forward(self, x, return_attention=False):
        attn_out, attn_w = self.attn(self.ln1(x), return_attention)
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x, attn_w

class DecoderOnlyTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.config = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.emb_dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_ln = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_weights:
            self.lm_head.weight = self.embedding.weight
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if m.bias is not None: torch.nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if m.padding_idx is not None: m.weight.data[m.padding_idx].zero_()
            elif isinstance(m, nn.LayerNorm):
                torch.nn.init.ones_(m.weight); torch.nn.init.zeros_(m.bias)
    
    def forward(self, input_ids, targets=None, return_attention=False):
        x = self.emb_dropout(self.embedding(input_ids))
        all_attn = [] if return_attention else None
        for block in self.blocks:
            x, aw = block(x, return_attention)
            if return_attention and aw is not None: all_attn.append(aw)
        logits = self.lm_head(self.final_ln(x))
        result = {"logits": logits}
        if targets is not None:
            result["loss"] = F.cross_entropy(logits.view(-1, self.config.vocab_size), targets.view(-1), ignore_index=self.config.pad_id)
        if return_attention: result["attention_weights"] = all_attn
        return result

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=128, temperature=1.0, top_k=0, repetition_penalty=1.0):
        self.eval(); gen = input_ids
        for _ in range(max_new_tokens):
            ctx = gen if gen.shape[1] <= self.config.max_seq_len else gen[:, -self.config.max_seq_len:]
            logits = self.forward(ctx)["logits"][:, -1, :]
            if repetition_penalty != 1.0:
                for b in range(input_ids.size(0)):
                    for token_id in set(gen[b].tolist()):
                        if logits[b, token_id] < 0: logits[b, token_id] *= repetition_penalty
                        else: logits[b, token_id] /= repetition_penalty
            if temperature == 0.0: nt = logits.argmax(dim=-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k > 0:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = float("-inf")
                nt = torch.multinomial(torch.softmax(logits, dim=-1), 1)
            gen = torch.cat([gen, nt], dim=1)
            if (nt == self.config.eos_id).all(): break
        return gen

In [4]:
# ==========================================
# DATA LOADING
# ==========================================
class TokenizedDataset(Dataset):
    def __init__(self, bin_path, seq_len=512):
        self.seq_len = seq_len
        self.data = np.memmap(bin_path, dtype=np.uint16, mode="r")
        self.n_samples = max(0, (len(self.data) - 1) // seq_len)
    def __len__(self): return self.n_samples
    def __getitem__(self, idx):
        start = idx * self.seq_len
        c = self.data[start:start+self.seq_len+1].astype(np.int64)
        return torch.from_numpy(c[:-1]), torch.from_numpy(c[1:])
    def get_total_tokens(self): return len(self.data)

def get_cosine_schedule_with_warmup(optimizer, warmup_steps, max_steps, min_lr, peak_lr):
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, max_steps - warmup_steps))
        return min_lr / peak_lr + 0.5 * (1.0 - min_lr / peak_lr) * (1.0 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def create_optimizer(model, weight_decay, learning_rate):
    param_dict = {pn: p for pn, p in model.named_parameters() if p.requires_grad}
    decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
    nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
    optim_groups = [
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": nodecay_params, "weight_decay": 0.0}
    ]
    return torch.optim.AdamW(optim_groups, lr=learning_rate, betas=(0.9, 0.95), eps=1e-8)

In [5]:
# ==========================================
# EVALUATION HELPER
# ==========================================
@torch.no_grad()
def evaluate(model, val_loader, device, max_batches=100):
    model.eval()
    total_loss, total_tokens, n_batches = 0.0, 0, 0
    for x, y in val_loader:
        if max_batches and n_batches >= max_batches: break
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            result = model(x, targets=y)
        non_pad = (y != PAD_ID).sum().item()
        total_loss += result["loss"].item() * non_pad
        total_tokens += non_pad; n_batches += 1
    avg_loss = total_loss / max(1, total_tokens)
    ppl = math.exp(avg_loss) if avg_loss < 20 else float("inf")
    model.train()
    return avg_loss, ppl

In [6]:
# ==========================================
# MAIN TRAINING
# ==========================================
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Using device: {device}")
    
    model = DecoderOnlyTransformer(config).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f"Model Parameters (No-RoPE): Total = {total_params:,}, Trainable = {trainable_params:,} (~{trainable_params/1e6:.2f}M)")
    
    train_ds = TokenizedDataset(TRAIN_BIN, MAX_SEQ_LEN)
    val_ds = TokenizedDataset(VAL_BIN, MAX_SEQ_LEN)
    logger.info(f"Dataset: Train chunks = {len(train_ds):,}, Val chunks = {len(val_ds):,}")
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, pin_memory=True)
    
    optimizer = create_optimizer(model, WEIGHT_DECAY, LR)
    scheduler = get_cosine_schedule_with_warmup(optimizer, WARMUP_STEPS, MAX_STEPS, MIN_LR, LR)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type=="cuda")
    
    step = 0
    best_val_loss = float("inf")
    train_iter = iter(train_loader)
    training_log = []
    
    logger.info("Starting training loop...")
    t0 = time.time()
    
    while step < MAX_STEPS:
        model.train()
        optimizer.zero_grad(set_to_none=True)
        accum_loss = 0.0
        
        for micro in range(GRAD_ACCUM):
            try:
                x, y = next(train_iter)
            except StopIteration:
                train_iter = iter(train_loader)
                x, y = next(train_iter)
                
            x, y = x.to(device), y.to(device)
            with torch.amp.autocast("cuda", dtype=torch.float16):
                res = model(x, targets=y)
                loss = res["loss"] / GRAD_ACCUM
            
            scaler.scale(loss).backward()
            accum_loss += loss.item() * GRAD_ACCUM
            
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        step += 1
        
        if step % LOG_INTERVAL == 0:
            dt = time.time() - t0
            tok_per_sec = (LOG_INTERVAL * GRAD_ACCUM * BATCH_SIZE * MAX_SEQ_LEN) / dt if dt > 0 else 0
            curr_lr = optimizer.param_groups[0]["lr"]
            logger.info(f"Step {step:5d}/{MAX_STEPS} | Train Loss: {accum_loss:.4f} | LR: {curr_lr:.2e} | Tok/s: {tok_per_sec:,.0f}")
            t0 = time.time()
            
        if step % EVAL_INTERVAL == 0 or step == MAX_STEPS:
            val_loss, val_ppl = evaluate(model, val_loader, device, max_batches=EVAL_MAX_BATCHES)
            curr_lr = optimizer.param_groups[0]["lr"]
            logger.info(f"*** EVAL Step {step} | Val Loss: {val_loss:.4f} | Val PPL: {val_ppl:.2f} ***")
            
            entry = {
                "step": step,
                "train_loss": round(accum_loss, 4),
                "val_loss": round(val_loss, 4),
                "val_ppl": round(val_ppl, 2),
                "lr": curr_lr
            }
            training_log.append(entry)
            with open(os.path.join(OUTPUT_DIR, "training_log.json"), "w") as f:
                json.dump(training_log, f, indent=2)
                
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_ckpt = {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "scaler_state_dict": scaler.state_dict(),
                    "step": step,
                    "val_loss": val_loss,
                    "config": config.__dict__
                }
                torch.save(best_ckpt, os.path.join(OUTPUT_DIR, "best_model.pt"))
                logger.info(f">>> Saved new best model at step {step} (val_loss={val_loss:.4f})")
                
        if step % SAVE_INTERVAL == 0:
            ckpt = {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "step": step,
                "val_loss": accum_loss,
                "config": config.__dict__
            }
            torch.save(ckpt, os.path.join(OUTPUT_DIR, f"checkpoint_step_{step}.pt"))
            logger.info(f"Saved periodic checkpoint at step {step}")
            
    logger.info("Training complete!")

if __name__ == "__main__":
    main()

[2026-09-13 12:34:08,174] INFO — Using device: cuda
[2026-09-13 12:34:08,957] INFO — Model Parameters (No-RoPE): Total = 25,084,928, Trainable = 25,084,928 (~25.08M)
[2026-09-13 12:34:08,990] INFO — Dataset: Train chunks = 1,551,841, Val chunks = 32,268
[2026-09-13 12:34:13,941] INFO — Starting training loop...
[2026-09-13 12:35:24,504] INFO — Step    50/16000 | Train Loss: 34.7397 | LR: 7.50e-06 | Tok/s: 23,219
[2026-09-13 12:36:20,598] INFO — Step   100/16000 | Train Loss: 32.5223 | LR: 1.50e-05 | Tok/s: 29,209
[2026-09-13 12:37:19,949] INFO — Step   150/16000 | Train Loss: 31.0480 | LR: 2.25e-05 | Tok/s: 27,606
[2026-09-13 12:38:12,922] INFO — Step   200/16000 | Train Loss: 28.8197 | LR: 3.00e-05 | Tok/s: 30,930
[2026-09-13 12:39:03,927] INFO — Step   250/16000 | Train Loss: 27.5068 | LR: 3.75e-05 | Tok/s: 32,123
[2026-09-13 12:39:54,663] INFO — Step   300/16000 | Train Loss: 26.3362 | LR: 4.50e-05 | Tok/s: 32,293
[2026-09-13 12:40:43,540] INFO — Step   350/16000 | Train Loss: 25.41